# 🛣️ RoadGuard: Deep Learning Road Damage Dataset Expansion & Fine-Tuning
### Ingesting Lorenzo Arcioni Kaggle Dataset (Potholes, Cracks, Manholes) with RDD2022 Baseline

**Target Dataset:** [Lorenzo Arcioni Kaggle Road Damage Dataset](https://www.kaggle.com/datasets/lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes)
- **Publication:** *Scientific Reports* (Nature Portfolio, 2026), DOI: [10.1038/s41598-026-46679-4](https://doi.org/10.1038/s41598-026-46679-4)
- **License:** Creative Commons Attribution 4.0 International (CC BY 4.0)
- **Capture Devices:** In-vehicle GoPro HERO7 and handheld Samsung Galaxy A14 smartphone
- **Target Objectives:**
  1. Resolve large foreground pothole cavitation recall ($>200\text{px}$ craters) missed by $P_5$ receptive fields.
  2. Ingest manhole annotations as negative background features to eliminate false-positive pothole triggers on utility covers.
  3. Maintain strict separation of RDD2022 crack classes (D00, D10, D20) by safely excluding generic cracks.

**Hardware Requirement:** Google Colab Free T4 GPU (Runtime -> Change runtime type -> T4 GPU).

## 1. GPU Environment & Dependency Verification

In [ ]:
# Verify GPU status
!nvidia-smi

# Install Ultralytics, Kaggle CLI, and image processing tools
!pip install -q ultralytics kaggle opencv-python-headless pyyaml matplotlib seaborn

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU Device: {torch.cuda.get_device_name(0)}")

## 2. Kaggle API Authentication & Dataset Download
Upload your `kaggle.json` API token (from Kaggle Account Settings -> 'Create New Token') OR set environment variables.

In [ ]:
import os
import getpass
from pathlib import Path
from google.colab import files

kaggle_dir = Path.home() / ".kaggle"
kaggle_dir.mkdir(parents=True, exist_ok=True)
kaggle_json = kaggle_dir / "kaggle.json"

if not kaggle_json.exists():
    print("Choose authentication method:")
    print("Option 1: Upload kaggle.json file")
    print("Option 2: Input Kaggle Username & Key")
    choice = input("Enter '1' to upload file or '2' to enter credentials: ").strip()
    
    if choice == '1':
        print("Please upload your kaggle.json:")
        uploaded = files.upload()
        for fn in uploaded.keys():
            if fn == 'kaggle.json':
                with open(kaggle_json, 'wb') as f:
                    f.write(uploaded[fn])
    else:
        user = input("Kaggle Username: ").strip()
        key = getpass.getpass("Kaggle API Key: ").strip()
        with open(kaggle_json, 'w') as f:
            f.write(f'{{"username":"{user}","key":"{key}"}}')

!chmod 600 ~/.kaggle/kaggle.json
print("Kaggle API configured successfully!")

# Download the Lorenzo Arcioni dataset
ARCIONI_DIR = Path("/content/arcioni_raw")
ARCIONI_DIR.mkdir(parents=True, exist_ok=True)

print("Downloading 'lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes'...")
!kaggle datasets download -d lorenzoarcioni/road-damage-dataset-potholes-cracks-and-manholes -p /content/arcioni_raw --unzip
print("Download and extraction completed!")

## 3. Dataset Census & Bounding-Box Quality Inspection
Inspect image dimensions, aspect ratios, annotation distribution, and bounding box integrity.

In [ ]:
import hashlib
from collections import Counter
import cv2
import matplotlib.pyplot as plt

def inspect_dataset(raw_dir):
    img_paths = list(raw_dir.rglob("*.jpg")) + list(raw_dir.rglob("*.jpeg")) + list(raw_dir.rglob("*.png"))
    print(f"Total Images Discovered: {len(img_paths)}")
    
    ann_counts = Counter()
    resolutions = Counter()
    unique_hashes = set()
    duplicates = 0
    
    for p in img_paths:
        # MD5 check
        h = hashlib.md5(p.read_bytes()).hexdigest()
        if h in unique_hashes:
            duplicates += 1
        else:
            unique_hashes.add(h)
            
        # Check label
        lbl = p.with_suffix(".txt")
        if not lbl.exists():
            # Check in parallel labels dir
            parts = list(p.parts)
            if "images" in parts:
                idx = len(parts) - 1 - parts[::-1].index("images")
                parts[idx] = "labels"
                cand = Path(*parts).with_suffix(".txt")
                if cand.exists(): lbl = cand
                
        if lbl.exists():
            for line in lbl.read_text().strip().split("\n"):
                parts = line.strip().split()
                if len(parts) >= 5:
                    ann_counts[int(parts[0])] += 1
                    
    print(f"Unique Images: {len(unique_hashes)} (Duplicates: {duplicates})")
    print("\nAnnotation Breakdown (Lorenzo Arcioni Taxonomy):")
    print(f"  Class 0 (Pothole): {ann_counts[0]} instances")
    print(f"  Class 1 (Crack)  : {ann_counts[1]} instances")
    print(f"  Class 2 (Manhole): {ann_counts[2]} instances")
    return img_paths

img_paths = inspect_dataset(ARCIONI_DIR)

## 4. Download Official RDD2022 Baseline Subsets (Japan + India)

In [ ]:
RDD_RAW = Path("/content/rdd2022_raw")
RDD_RAW.mkdir(parents=True, exist_ok=True)

URLS = {
    "Japan": "https://bigdata-cup.s3.ap-northeast-1.amazonaws.com/2022/CRDDC2022/RDD2022_Japan.tar.gz",
    "India": "https://bigdata-cup.s3.ap-northeast-1.amazonaws.com/2022/CRDDC2022/RDD2022_India.tar.gz"
}

for country, url in URLS.items():
    archive = RDD_RAW / f"{country}.tar.gz"
    if not archive.exists():
        print(f"Downloading RDD2022 {country} subset...")
        !wget -q --show-progress {url} -O {archive}
        print(f"Extracting {country}...")
        !tar -xzf {archive} -C {RDD_RAW}

print("RDD2022 Baseline subsets ready.")

## 5. Convert RDD2022 VOC XML Annotations to YOLO Format

In [ ]:
import xml.etree.ElementTree as ET
import shutil

RDD_YOLO = Path("/content/rdd2022_processed")
train_img_dst = RDD_YOLO / "train" / "images"
train_lbl_dst = RDD_YOLO / "train" / "labels"
val_img_dst = RDD_YOLO / "val" / "images"
val_lbl_dst = RDD_YOLO / "val" / "labels"

for d in [train_img_dst, train_lbl_dst, val_img_dst, val_lbl_dst]:
    d.mkdir(parents=True, exist_ok=True)

RDD_CLASS_MAP = {"D00": 0, "D10": 1, "D20": 2, "D40": 3}

def convert_voc_xml(xml_path, img_w, img_h):
    tree = ET.parse(xml_path)
    root = tree.getroot()
    lines = []
    for obj in root.findall("object"):
        cls_name = obj.find("name").text.strip()
        if cls_name not in RDD_CLASS_MAP:
            continue
        cls_id = RDD_CLASS_MAP[cls_name]
        bnd = obj.find("bndbox")
        xmin = float(bnd.find("xmin").text)
        ymin = float(bnd.find("ymin").text)
        xmax = float(bnd.find("xmax").text)
        ymax = float(bnd.find("ymax").text)
        
        x_c = ((xmin + xmax) / 2.0) / img_w
        y_c = ((ymin + ymax) / 2.0) / img_h
        w = (xmax - xmin) / img_w
        h = (ymax - ymin) / img_h
        lines.append(f"{cls_id} {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}")
    return lines

print("Converting RDD2022 XMLs to YOLO...")
# Process countries
import random
random.seed(42)
rdd_count = 0
for country_dir in RDD_RAW.glob("*"):
    if not country_dir.is_dir(): continue
    xmls_dir = country_dir / "annotations" / "xmls"
    imgs_dir = country_dir / "images"
    if not (xmls_dir.exists() and imgs_dir.exists()): continue
    
    for xml_p in xmls_dir.glob("*.xml"):
        img_p = imgs_dir / f"{xml_p.stem}.jpg"
        if not img_p.exists(): continue
        
        # Read dims
        img = cv2.imread(str(img_p))
        if img is None: continue
        ih, iw = img.shape[:2]
        
        lines = convert_voc_xml(xml_p, iw, ih)
        if not lines: continue
        
        is_val = random.random() < 0.15
        tgt_img = (val_img_dst if is_val else train_img_dst) / img_p.name
        tgt_lbl = (val_lbl_dst if is_val else train_lbl_dst) / f"{xml_p.stem}.txt"
        shutil.copy2(img_p, tgt_img)
        tgt_lbl.write_text("\n".join(lines) + "\n")
        rdd_count += 1

print(f"RDD2022 processing complete: {rdd_count} images ingested into {RDD_YOLO}.")

## 6. Build Strict Unified Dataset (Incorporating Lorenzo Arcioni Potholes & Manholes)
Executes the RoadGuard Dataset Combiner:
- Potholes (Class 0) mapped to Canonical Class 3 (`D40_Pothole`).
- Generic cracks (Class 1) **excluded** to prevent crack taxonomy pollution.
- Manholes (Class 2) converted into verified negative background frames.
- Sequential dashcam duplicates removed via MD5 hash.

In [ ]:
# Execute dataset combiner
UNIFIED_DIR = Path("/content/roadguard_unified")
u_train_img = UNIFIED_DIR / "train" / "images"
u_train_lbl = UNIFIED_DIR / "train" / "labels"
u_val_img = UNIFIED_DIR / "val" / "images"
u_val_lbl = UNIFIED_DIR / "val" / "labels"

for d in [u_train_img, u_train_lbl, u_val_img, u_val_lbl]:
    d.mkdir(parents=True, exist_ok=True)

# 1. Copy RDD2022 processed
print("Copying RDD2022 into Unified Dataset...")
for p in train_img_dst.glob("*.*"):
    shutil.copy2(p, u_train_img / f"rdd_{p.name}")
    l = train_lbl_dst / f"{p.stem}.txt"
    if l.exists(): shutil.copy2(l, u_train_lbl / f"rdd_{p.stem}.txt")
for p in val_img_dst.glob("*.*"):
    shutil.copy2(p, u_val_img / f"rdd_{p.name}")
    l = val_lbl_dst / f"{p.stem}.txt"
    if l.exists(): shutil.copy2(l, u_val_lbl / f"rdd_{p.stem}.txt")

# 2. Ingest Lorenzo Arcioni Potholes & Manholes
print("Ingesting Lorenzo Arcioni Potholes & Manholes...")
seen_hashes = set()
arc_potholes = 0
arc_manholes = 0
arc_cracks_excluded = 0

arc_images = list(ARCIONI_DIR.rglob("*.jpg")) + list(ARCIONI_DIR.rglob("*.png"))
for img_p in arc_images:
    h = hashlib.md5(img_p.read_bytes()).hexdigest()
    if h in seen_hashes: continue
    seen_hashes.add(h)
    
    lbl_p = img_p.with_suffix(".txt")
    if not lbl_p.exists():
        parts = list(img_p.parts)
        if "images" in parts:
            idx = len(parts) - 1 - parts[::-1].index("images")
            parts[idx] = "labels"
            cand = Path(*parts).with_suffix(".txt")
            if cand.exists(): lbl_p = cand
    if not lbl_p.exists(): continue
    
    pothole_boxes = []
    has_crack = False
    has_manhole = False
    
    for line in lbl_p.read_text().strip().split("\n"):
        parts = line.strip().split()
        if len(parts) >= 5:
            c = int(parts[0])
            if c == 0: pothole_boxes.append(parts[1:5])
            elif c == 1: has_crack = True
            elif c == 2: has_manhole = True
            
    # Strict Exclusions:
    if has_crack:
        arc_cracks_excluded += 1
        continue  # Protect crack taxonomy
        
    is_val = random.random() < 0.15
    dst_i = u_val_img if is_val else u_train_img
    dst_l = u_val_lbl if is_val else u_train_lbl
    
    if pothole_boxes:
        shutil.copy2(img_p, dst_i / f"arc_{img_p.name}")
        with open(dst_l / f"arc_{img_p.stem}.txt", "w") as f:
            for b in pothole_boxes:
                f.write(f"3 {' '.join(b)}\n")
        arc_potholes += len(pothole_boxes)
    elif has_manhole and not pothole_boxes:
        # Negative background sample
        shutil.copy2(img_p, dst_i / f"arc_bg_{img_p.name}")
        with open(dst_l / f"arc_bg_{img_p.stem}.txt", "w") as f:
            pass
        arc_manholes += 1

# 3. Write data.yaml
data_yaml = f"""
path: /content/roadguard_unified
train: train/images
val: val/images
nc: 4
names:
  0: D00_Longitudinal_Crack
  1: D10_Transverse_Crack
  2: D20_Alligator_Crack
  3: D40_Pothole
"""
with open(UNIFIED_DIR / "data.yaml", "w") as f:
    f.write(data_yaml.strip())

print("\nUnified RoadGuard Dataset Created:")
print(f"  Total Training Images  : {len(list(u_train_img.glob('*.*')))}")
print(f"  Total Validation Images: {len(list(u_val_img.glob('*.*')))}")
print(f"  Arcioni Potholes Mapped: {arc_potholes}")
print(f"  Manhole Negative BGs   : {arc_manholes}")
print(f"  Generic Cracks Excluded: {arc_cracks_excluded}")

## 7. Model Training Configuration (DO NOT RUN AUTOMATICALLY)
> **Note:** Inspect the parameters below. Execute only when ready to start 50-epoch GPU fine-tuning.

In [ ]:
# Fine-Tuning Execution Cell (Execute manually when ready)
from ultralytics import YOLO

# Initialize model with YOLOv8s
model = YOLO("yolov8s.pt")

# Train for 50 epochs with multi-scale, lighting, and perspective invariance
results = model.train(
    data="/content/roadguard_unified/data.yaml",
    epochs=50,
    batch=16,
    imgsz=640,
    device=0,
    project="/content/runs/train",
    name="roadguard_arcioni_rdd2022",
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    scale=0.5,
    mosaic=1.0,
    mixup=0.15,
    patience=15,
    save=True
)

## 8. Quantitative Evaluation on Held-Out Validation Benchmark

In [ ]:
# Validate fine-tuned model
metrics = model.val()
print("\n--- Fine-Tuned Model Performance ---")
print(f"All-Classes mAP@0.5     : {metrics.box.map50:.4f}")
print(f"All-Classes mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"Class-wise mAP@0.5      : {metrics.box.maps}")

## 9. Export & Download Fine-Tuned Weights (`best.pt`)

In [ ]:
from google.colab import files
best_pt = Path("/content/runs/train/roadguard_arcioni_rdd2022/weights/best.pt")
if best_pt.exists():
    print("Downloading fine-tuned weights best.pt to your computer...")
    files.download(str(best_pt))
else:
    print("Trained weights not found. Complete step 7 first.")